# AI-Assisted Diagnosis of Cutaneous Melanoma
## Complete 600-case Attention U-Net → lesion ROI → EfficientNetB0 → Grad-CAM + ROI input comparison

This is one self-contained Google Colab notebook. Use **Runtime → Run all** from the beginning. The three-input experiment is in Part F, after the original pipeline and results. Its comparison and threshold tables use training/validation data only.


## 0. Run in Colab

Choose Runtime → Change runtime type → GPU, then Runtime → Restart session → Run all. The ISIC ZIP is downloaded to temporary storage and 600 image/mask pairs are extracted. Model checkpoints and tables can optionally be saved to Drive. All results must be newly computed.

## 2. Imports and reproducibility

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Global configuration

In [ ]:
PREVIOUS_SELECTED_CASES = 300
TOTAL_CASES = 600
MODELLING_CASES = 500
FINAL_TEST_CASES = 100
MODEL_TRAIN_CASES = 400
MODEL_VAL_CASES = 100
SEG_IMAGE_SIZE = (256, 256)
CLS_IMAGE_SIZE = (224, 224)
SEG_BATCH_SIZE = 4
CLS_BATCH_SIZE = 16
SEG_EPOCHS = 40
CLS_HEAD_EPOCHS = 15
CLS_FINE_TUNE_EPOCHS = 10
SEG_LEARNING_RATE = 1e-3
CLS_HEAD_LR = 1e-3
CLS_FINE_TUNE_LR = 1e-5
SEG_THRESHOLD = 0.50
CLS_THRESHOLD = 0.50
DO_FINE_TUNING = True
RUNTIME_ROOT = Path('/content/melanoma_600_streamlined')
DOWNLOAD_DIR = RUNTIME_ROOT / 'downloads'
DATA_DIR = RUNTIME_ROOT / 'selected_600'
ROI_DIR = RUNTIME_ROOT / 'roi_images'
SAVE_RESULTS_TO_DRIVE = True
for folder in [RUNTIME_ROOT, DOWNLOAD_DIR, DATA_DIR, ROI_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('Fresh split: 400 train + 100 validation + 100 final test')

## 4. Optional Google Drive output folder

In [ ]:
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        OUTPUT_DIR = Path(
            "/content/drive/MyDrive/Melanoma_AI_AttentionUNet_600_Streamlined_Results"
        )
    except Exception as exc:
        print("Drive mount unavailable. Falling back to /content.")
        print("Reason:", exc)
        OUTPUT_DIR = RUNTIME_ROOT / "results"
else:
    OUTPUT_DIR = RUNTIME_ROOT / "results"

MODELS_DIR = OUTPUT_DIR / "models"
RESULTS_DIR = OUTPUT_DIR / "results"
SPLITS_DIR = OUTPUT_DIR / "splits"

for folder in [OUTPUT_DIR, MODELS_DIR, RESULTS_DIR, SPLITS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", OUTPUT_DIR)

## 5. Official ISIC 2016 Task 3B data

The official **ISIC 2016 Task 3B: Segmented Lesion Classification** training data contains:

- 900 dermoscopic JPEG images,
- 900 matching expert segmentation masks,
- 900 benign/malignant labels.

This notebook downloads the official training ZIP and CSV into temporary Colab storage, reserves a new 100-case final test group from previously unused cases, extracts 600 selected image/mask pairs, and deletes the large ZIP afterwards.

In [ ]:
DATA_ZIP_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_Data.zip"
)

LABEL_CSV_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"
)

DATA_ZIP = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_Data.zip"
LABEL_CSV = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"


def download_file(url, destination, chunk_mb=8):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 100:
        print("Already downloaded:", destination.name)
        return destination

    tmp = destination.with_suffix(destination.suffix + ".part")
    start = tmp.stat().st_size if tmp.exists() else 0
    headers = {"Range": f"bytes={start}-"} if start else {}

    print("Downloading:", destination.name)

    with requests.get(
        url,
        stream=True,
        headers=headers,
        timeout=(30, 120),
    ) as response:
        response.raise_for_status()

        if start and response.status_code != 206:
            start = 0
            tmp.unlink(missing_ok=True)

        mode = "ab" if start else "wb"

        with open(tmp, mode) as handle:
            downloaded = start
            last_print = time.time()

            for chunk in response.iter_content(
                chunk_size=chunk_mb * 1024 * 1024
            ):
                if not chunk:
                    continue

                handle.write(chunk)
                downloaded += len(chunk)

                if time.time() - last_print > 5:
                    print(
                        f"  downloaded {downloaded / (1024**2):.1f} MB"
                    )
                    last_print = time.time()

    tmp.replace(destination)
    print("Finished:", destination)
    return destination


download_file(LABEL_CSV_URL, LABEL_CSV)
download_file(DATA_ZIP_URL, DATA_ZIP)

assert LABEL_CSV.exists(), "Ground-truth CSV download failed."
assert DATA_ZIP.exists(), "Training ZIP download failed."
assert zipfile.is_zipfile(DATA_ZIP), (
    "Downloaded training file is not a valid ZIP."
)

print("Official ISIC files are ready in temporary Colab storage.")

## 6. Load labels and reserve a new final test group

In [ ]:
labels_df = pd.read_csv(LABEL_CSV, header=None, names=['image_id', 'diagnosis'])
labels_df['diagnosis'] = labels_df['diagnosis'].astype(str).str.strip().str.lower()
labels_df['label'] = labels_df['diagnosis'].map({'benign': 0, 'malignant': 1})
assert len(labels_df) == 900 and labels_df['image_id'].is_unique
assert labels_df['label'].notna().all()

# Reproduce the earlier selection exactly. Its former test cases are now
# development data; none appears in this version's final test group.
historical_300, untouched_600 = train_test_split(
    labels_df, train_size=PREVIOUS_SELECTED_CASES,
    stratify=labels_df['label'], random_state=SEED,
)
new_test_100, additional_500 = train_test_split(
    untouched_600, train_size=FINAL_TEST_CASES,
    stratify=untouched_600['label'], random_state=SEED + 1,
)
extra_200, unused_300 = train_test_split(
    additional_500, train_size=200,
    stratify=additional_500['label'], random_state=SEED + 2,
)
selected_cases = pd.concat([historical_300, extra_200, new_test_100],
                           ignore_index=True)
assert len(selected_cases) == TOTAL_CASES
assert set(new_test_100.image_id).isdisjoint(set(historical_300.image_id))
print('Selected class counts:')
display(selected_cases['diagnosis'].value_counts().to_frame('count'))

## 7. Split 500 development cases into 400 train / 100 validation

In [ ]:
model_df = pd.concat([historical_300, extra_200], ignore_index=True)
test_df = new_test_100.reset_index(drop=True)
train_df, val_df = train_test_split(
    model_df, train_size=MODEL_TRAIN_CASES, test_size=MODEL_VAL_CASES,
    stratify=model_df['label'], random_state=SEED + 3,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
assert len(train_df) == MODEL_TRAIN_CASES and len(val_df) == MODEL_VAL_CASES
assert len(test_df) == FINAL_TEST_CASES
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))
assert set(model_df.image_id).isdisjoint(set(test_df.image_id))
for name, frame in [('train', train_df), ('validation', val_df),
                    ('final_test', test_df)]:
    print(f'{name}: n={len(frame)}')
    display(frame['diagnosis'].value_counts().to_frame('count'))
    frame[['image_id', 'diagnosis', 'label']].to_csv(
        SPLITS_DIR / f'{name}_{len(frame)}.csv', index=False)
print('Split files:', SPLITS_DIR)

## 8. Extract the selected 600 images and masks

In [ ]:
selected_ids = set(selected_cases["image_id"])

with zipfile.ZipFile(DATA_ZIP, "r") as archive:
    file_members = [
        member
        for member in archive.namelist()
        if not member.endswith("/")
    ]

    by_basename = {
        Path(member).name: member
        for member in file_members
    }

    missing_images = []
    missing_masks = []
    extracted_records = []

    for image_id in sorted(selected_ids):
        image_base = f"{image_id}.jpg"
        mask_base = f"{image_id}_Segmentation.png"

        image_member = by_basename.get(image_base)
        mask_member = by_basename.get(mask_base)

        if image_member is None:
            missing_images.append(image_id)
            continue

        if mask_member is None:
            missing_masks.append(image_id)
            continue

        image_output = DATA_DIR / image_base
        mask_output = DATA_DIR / mask_base

        if not image_output.exists():
            with archive.open(image_member) as src:
                with open(image_output, "wb") as dst_file:
                    shutil.copyfileobj(src, dst_file)

        if not mask_output.exists():
            with archive.open(mask_member) as src:
                with open(mask_output, "wb") as dst_file:
                    shutil.copyfileobj(src, dst_file)

        extracted_records.append(
            {
                "image_id": image_id,
                "image_path": str(image_output),
                "mask_path": str(mask_output),
            }
        )

assert not missing_images, (
    f"Missing selected images: {missing_images[:10]}"
)
assert not missing_masks, (
    f"Missing selected masks: {missing_masks[:10]}"
)
assert len(extracted_records) == TOTAL_CASES, (
    f"Expected {TOTAL_CASES} extracted pairs, found "
    f"{len(extracted_records)}."
)

path_df = pd.DataFrame(extracted_records)


def add_paths(frame):
    result = frame.merge(
        path_df,
        on="image_id",
        how="left",
    )
    assert result["image_path"].notna().all()
    assert result["mask_path"].notna().all()
    return result


train_df = add_paths(train_df)
val_df = add_paths(val_df)
model_df = add_paths(model_df)
test_df = add_paths(test_df)
selected_cases = add_paths(selected_cases)

if DATA_ZIP.exists():
    DATA_ZIP.unlink()
    print(
        "Deleted large temporary ZIP:",
        DATA_ZIP.name,
    )

print(
    "Extracted selected image-mask pairs:",
    len(selected_cases),
)
print("Selected data directory:", DATA_DIR)


## 9. Inspect images, expert masks and their alignment

In [ ]:
def segmentation_overlay(image_pil, mask_binary, alpha=0.30):
    image = np.asarray(image_pil.convert('RGB'), dtype=np.float32).copy()
    binary = (np.asarray(mask_binary) > 0).astype(np.uint8)
    mask = Image.fromarray(binary * 255).resize(image_pil.size, Image.Resampling.NEAREST)
    selected = np.asarray(mask) > 127
    image[selected] = (1 - alpha) * image[selected] + alpha * np.array([255, 0, 0])
    return Image.fromarray(np.uint8(np.clip(image, 0, 255)))
def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_mask(path):
    with Image.open(path) as image:
        return image.convert("L").copy()


sample_rows = pd.concat(
    [
        train_df[train_df["label"] == 0].head(2),
        train_df[train_df["label"] == 1].head(2),
    ]
).reset_index(drop=True)

fig, axes = plt.subplots(
    len(sample_rows),
    3,
    figsize=(12, 4 * len(sample_rows)),
)

if len(sample_rows) == 1:
    axes = np.array([axes])

for row_index, row in sample_rows.iterrows():
    image = load_rgb(row["image_path"])
    mask = load_mask(row["mask_path"])

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(
        f"{row['image_id']} | {row['diagnosis']}"
    )
    axes[row_index, 0].axis("off")

    axes[row_index, 1].imshow(mask, cmap="gray")
    axes[row_index, 1].set_title("Expert mask")
    axes[row_index, 1].axis("off")

    axes[row_index, 2].imshow(segmentation_overlay(image, np.asarray(mask) > 127))
    axes[row_index, 2].set_title("Mask overlay")
    axes[row_index, 2].axis("off")

plt.tight_layout()
plt.show()

# PART A — LESION SEGMENTATION

This version focuses only on **Attention U-Net**.

The model uses:

- 160 training cases,
- 40 validation cases,
- paired image/mask augmentation,
- Dice + IoU evaluation,
- early stopping,
- learning-rate reduction,
- and the final 100 cases remain untouched until held-out testing.

Attention gates are applied to the encoder-to-decoder skip connections to emphasise lesion-relevant features.


## 10. `tf.data` segmentation pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_segmentation_pair(
    image_path,
    mask_path,
):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )
    image = tf.image.resize(
        image,
        SEG_IMAGE_SIZE,
        method="bilinear",
    )
    image = tf.cast(image, tf.float32) / 255.0

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(
        mask_bytes,
        channels=1,
    )
    mask = tf.image.resize(
        mask,
        SEG_IMAGE_SIZE,
        method="nearest",
    )
    mask = tf.cast(mask > 127, tf.float32)

    return image, mask


def paired_segmentation_augmentation(
    image,
    mask,
):
    seed1 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )
    seed2 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )

    image = tf.image.stateless_random_flip_left_right(
        image,
        seed1,
    )
    mask = tf.image.stateless_random_flip_left_right(
        mask,
        seed1,
    )

    image = tf.image.stateless_random_flip_up_down(
        image,
        seed2,
    )
    mask = tf.image.stateless_random_flip_up_down(
        mask,
        seed2,
    )

    # Same 0/90/180/270 degree rotation for image and expert mask.
    quarter_turns = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, quarter_turns)
    mask = tf.image.rot90(mask, quarter_turns)

    image = tf.image.random_brightness(
        image,
        max_delta=0.06,
    )
    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10,
    )
    image = tf.clip_by_value(
        image,
        0.0,
        1.0,
    )

    return image, mask


def make_seg_dataset(
    frame,
    training=False,
):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["image_path"].astype(str).values,
            frame["mask_path"].astype(str).values,
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_segmentation_pair,
        num_parallel_calls=AUTOTUNE,
    )

    if training:
        ds = ds.map(
            paired_segmentation_augmentation,
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(SEG_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_seg_ds = make_seg_dataset(
    train_df,
    training=True,
)
val_seg_ds = make_seg_dataset(
    val_df,
    training=False,
)
test_seg_ds = make_seg_dataset(
    test_df,
    training=False,
)

images_batch, masks_batch = next(
    iter(train_seg_ds)
)

print(
    "Image batch shape:",
    images_batch.shape,
)
print(
    "Mask batch shape:",
    masks_batch.shape,
)
print(
    "Image range:",
    float(tf.reduce_min(images_batch)),
    "to",
    float(tf.reduce_max(images_batch)),
)
print(
    "Mask values:",
    np.unique(masks_batch.numpy()),
)

## 11. Dice, IoU/Jaccard and segmentation loss

In [ ]:
EPS = 1e-7
SPATIAL_AXES = (1, 2, 3)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def soft_dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_coefficient(y_true, y_pred):
    # Hard per-image Dice averaged across the current batch.
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_loss(y_true, y_pred):
    return 1.0 - soft_dice_coefficient(y_true, y_pred)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((intersection + EPS) / (union + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def combined_segmentation_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)

## 12. Attention U-Net architecture


In [ ]:
def conv_block(
    x,
    filters,
    name,
):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv1",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu1",
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv2",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu2",
    )(x)

    return x


def attention_gate(
    skip,
    gating,
    inter_filters,
    name,
):
    theta = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_theta",
    )(skip)

    phi = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_phi",
    )(gating)

    x = layers.Add(
        name=f"{name}_add"
    )([theta, phi])

    x = layers.Activation(
        "relu",
        name=f"{name}_relu",
    )(x)

    psi = layers.Conv2D(
        1,
        1,
        padding="same",
        activation="sigmoid",
        name=f"{name}_psi",
    )(x)

    return layers.Multiply(
        name=f"{name}_weighted_skip"
    )([skip, psi])


def build_attention_unet(
    input_shape=(256, 256, 3),
):
    inputs = keras.Input(
        shape=input_shape,
        name="image",
    )

    c1 = conv_block(inputs, 32, "enc1")
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64, "enc2")
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128, "enc3")
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256, "enc4")
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(
        p4,
        512,
        "bottleneck",
    )

    def decoder_step(
        x,
        skip,
        filters,
        level,
    ):
        x = layers.Conv2DTranspose(
            filters,
            2,
            strides=2,
            padding="same",
            name=f"up{level}",
        )(x)

        skip = attention_gate(
            skip,
            x,
            max(filters // 2, 1),
            name=f"attention{level}",
        )

        x = layers.Concatenate(
            name=f"concat{level}"
        )([x, skip])

        return conv_block(
            x,
            filters,
            f"dec{level}",
        )

    d4 = decoder_step(bottleneck, c4, 256, 4)
    d3 = decoder_step(d4, c3, 128, 3)
    d2 = decoder_step(d3, c2, 64, 2)
    d1 = decoder_step(d2, c1, 32, 1)

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        dtype="float32",
        name="segmentation_mask",
    )(d1)

    return keras.Model(
        inputs,
        outputs,
        name="Attention_U_Net",
    )


attention_unet = build_attention_unet(
    input_shape=(
        *SEG_IMAGE_SIZE,
        3,
    )
)

print(
    "Attention U-Net parameters:",
    attention_unet.count_params(),
)

dummy = tf.zeros(
    (1, *SEG_IMAGE_SIZE, 3),
    dtype=tf.float32,
)

assert attention_unet(dummy).shape == (
    1,
    SEG_IMAGE_SIZE[0],
    SEG_IMAGE_SIZE[1],
    1,
)

print(
    "Attention U-Net passed "
    "the synthetic shape test."
)


## 13. Train Attention U-Net


In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "soft_dice_coefficient": soft_dice_coefficient,
    "dice_loss": dice_loss,
    "iou_coefficient": iou_coefficient,
    "combined_segmentation_loss": (
        combined_segmentation_loss
    ),
}


def compile_segmentation_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=SEG_LEARNING_RATE
        ),
        loss=combined_segmentation_loss,
        metrics=[
            dice_coefficient,
            iou_coefficient,
        ],
    )


def segmentation_callbacks(model_path):
    return [
        keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

attention_model = build_attention_unet(
    input_shape=(
        *SEG_IMAGE_SIZE,
        3,
    )
)

compile_segmentation_model(
    attention_model
)

ATTENTION_MODEL_PATH = (
    MODELS_DIR
    / "attention_unet_best.keras"
)

start_time = time.time()

attention_fit = attention_model.fit(
    train_seg_ds,
    validation_data=val_seg_ds,
    epochs=SEG_EPOCHS,
    callbacks=segmentation_callbacks(
        ATTENTION_MODEL_PATH
    ),
    verbose=1,
)

attention_training_seconds = (
    time.time() - start_time
)

attention_history = (
    attention_fit.history
)

selected_segmenter = (
    keras.models.load_model(
        ATTENTION_MODEL_PATH,
        custom_objects=CUSTOM_OBJECTS,
    )
)

attention_validation = (
    selected_segmenter.evaluate(
        val_seg_ds,
        verbose=0,
        return_dict=True,
    )
)

SELECTED_SEGMENTER_NAME = (
    "Attention U-Net"
)

SELECTED_SEGMENTER_PATH = (
    str(ATTENTION_MODEL_PATH)
)

print("\nAttention U-Net training finished.")
print(
    "Best checkpoint:",
    ATTENTION_MODEL_PATH,
)
print(
    "Validation Dice:",
    attention_validation[
        "dice_coefficient"
    ],
)
print(
    "Validation IoU:",
    attention_validation[
        "iou_coefficient"
    ],
)
print(
    "Parameters:",
    selected_segmenter.count_params(),
)

## 14. Lock the best Attention U-Net checkpoint


In [ ]:
best_validation_dice = float(
    np.max(
        attention_history[
            "val_dice_coefficient"
        ]
    )
)

best_validation_iou = float(
    np.max(
        attention_history[
            "val_iou_coefficient"
        ]
    )
)

attention_validation_summary = pd.DataFrame(
    {
        "Metric": [
            "Best Validation Dice (hard, batch mean)",
            "Best Validation IoU",
            "Validation Loss (best checkpoint)",
            "Parameters",
            "Training time (min)",
        ],
        "Value": [
            best_validation_dice,
            best_validation_iou,
            float(attention_validation["loss"]),
            int(selected_segmenter.count_params()),
            attention_training_seconds / 60.0,
        ],
    }
)

display(
    attention_validation_summary
)

attention_validation_summary.to_csv(
    RESULTS_DIR
    / "attention_unet_validation_summary.csv",
    index=False,
)

print(
    "Attention U-Net checkpoint locked "
    "using validation performance only."
)

# PART B — LESION ROI GENERATION

The selected segmentation model is now locked.

The **predicted mask** is used to create the ROI supplied to EfficientNetB0 so the classification stage matches the intended end-to-end pipeline.

## 16. Predicted mask and robust lesion ROI

Segmentation scores use the raw predicted mask. For each classifier crop, the largest connected region is selected so small stray pixels cannot expand the crop. The square ROI retains original RGB pixels with a little surrounding skin. Empty, implausible or boundary-limited masks fall back to the original image.

In [ ]:
from scipy import ndimage

def predict_mask_for_pil(model, image_pil):
    resized = image_pil.convert('RGB').resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    x = np.asarray(resized, dtype=np.float32) / 255.0
    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]
    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)
    return probability, binary

def extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.15):
    image = image_pil.convert('RGB')
    width, height = image.size
    mask = Image.fromarray(np.uint8(np.asarray(binary_mask) > 0) * 255)
    mask = np.asarray(mask.resize((width, height), Image.Resampling.NEAREST)) > 0

    def full_image():
        return image.copy(), (0, 0, width, height), True

    components, number = ndimage.label(mask)
    if number == 0:
        return full_image()
    sizes = np.bincount(components.ravel())
    sizes[0] = 0
    lesion = components == int(np.argmax(sizes))
    lesion_area = int(lesion.sum())
    if lesion_area < max(64, int(width * height * 0.001)) or lesion_area > width * height * 0.80:
        return full_image()

    ys, xs = np.where(lesion)
    box_w = int(xs.max() - xs.min() + 1)
    box_h = int(ys.max() - ys.min() + 1)
    side = int(np.ceil(max(box_w, box_h) * (1 + 2 * padding_fraction)))
    if side > min(width, height) or side < 8:
        return full_image()

    cx = (int(xs.min()) + int(xs.max()) + 1) / 2
    cy = (int(ys.min()) + int(ys.max()) + 1) / 2
    x0 = int(np.clip(round(cx - side / 2), 0, width - side))
    y0 = int(np.clip(round(cy - side / 2), 0, height - side))
    box = (x0, y0, x0 + side, y0 + side)
    return image.crop(box), box, False

## 17. Generate RGB lesion ROIs for train, validation and test cases

In [ ]:
ROI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

roi_records = []

for index, row in selected_cases.iterrows():
    image_pil = load_rgb(
        row["image_path"]
    )

    (
        probability_mask,
        binary_mask,
    ) = predict_mask_for_pil(
        selected_segmenter,
        image_pil,
    )

    (
        roi,
        bbox,
        fallback,
    ) = extract_roi_from_binary_mask(
        image_pil,
        binary_mask,
        padding_fraction=0.15,
    )

    roi_path = (
        ROI_DIR
        / f"{row['image_id']}_ROI.jpg"
    )

    roi.convert("RGB").save(
        roi_path,
        quality=95,
    )

    roi_records.append(
        {
            "image_id": row["image_id"],
            "roi_path": str(roi_path),
            "roi_fallback": bool(
                fallback
            ),
            "bbox": json.dumps(
                [
                    int(value)
                    for value
                    in bbox
                ]
            ),
        }
    )

    if (
        index + 1
    ) % 25 == 0:
        print(
            f"Generated "
            f"{index + 1}/{TOTAL_CASES} ROIs"
        )

roi_df = pd.DataFrame(
    roi_records
)


def attach_rois(frame):
    result = frame.merge(
        roi_df,
        on="image_id",
        how="left",
    )

    assert (
        result["roi_path"]
        .notna()
        .all()
    )

    return result


train_df = attach_rois(
    train_df
)
val_df = attach_rois(
    val_df
)
model_df = attach_rois(
    model_df
)
test_df = attach_rois(
    test_df
)

print("ROI generation complete.")
for split_name, split_frame in [('train', train_df), ('validation', val_df),
                                ('final_test', test_df)]:
    split_fallbacks = roi_df.set_index('image_id').loc[
        split_frame['image_id'], 'roi_fallback'
    ].sum()
    print(f'{split_name} fallback ROIs: {int(split_fallbacks)}/{len(split_frame)}')
print(
    "Fallback ROIs:",
    int(
        roi_df[
            "roi_fallback"
        ].sum()
    ),
    f"out of {TOTAL_CASES}",
)

## 18. Visualise original → predicted mask → ROI

In [ ]:
example = val_df.iloc[0]

image_pil = load_rgb(
    example["image_path"]
)

(
    probability_mask,
    binary_mask,
) = predict_mask_for_pil(
    selected_segmenter,
    image_pil,
)

roi_pil = load_rgb(
    example["roi_path"]
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4),
)

axes[0].imshow(
    image_pil
)
axes[0].set_title(
    "Original"
)
axes[0].axis(
    "off"
)

axes[1].imshow(
    binary_mask,
    cmap="gray",
)
axes[1].set_title(
    "Predicted lesion mask"
)
axes[1].axis(
    "off"
)

axes[2].imshow(
    roi_pil
)
axes[2].set_title(
    "Lesion ROI"
)
axes[2].axis(
    "off"
)

plt.tight_layout()
plt.show()

# PART C — EFFICIENTNETB0 CLASSIFICATION

Class 0 = benign and class 1 = melanoma. The classifier receives an original RGB lesion crop. Class weights address imbalance, and the frozen ImageNet backbone may be fine-tuned at a low learning rate. Fallback images retain their original dimensions.

## 19. `tf.data` classification pipeline

In [ ]:
classification_augmentation = (
    keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal_and_vertical",
                seed=SEED,
            ),
            layers.RandomRotation(
                0.05,
                seed=SEED,
            ),
            layers.RandomContrast(
                0.10,
                seed=SEED,
            ),
        ],
        name=(
            "classification_augmentation"
        ),
    )
)


def decode_classification_image(
    path,
    label,
):
    image_bytes = tf.io.read_file(
        path
    )

    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )

    image = tf.image.resize(
        image,
        CLS_IMAGE_SIZE,
        method="bilinear",
    )

    # EfficientNetB0 in Keras contains
    # its own input Rescaling layer.
    # Keep pixels in the 0-255 range.
    image = tf.cast(
        image,
        tf.float32,
    )

    label = tf.cast(
        label,
        tf.float32,
    )

    return image, label


def make_cls_dataset(
    frame,
    training=False,
):
    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                frame[
                    "roi_path"
                ]
                .astype(str)
                .values,
                frame[
                    "label"
                ]
                .astype(
                    np.float32
                )
                .values,
            )
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_classification_image,
        num_parallel_calls=AUTOTUNE,
    )

    ds = ds.batch(
        CLS_BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_cls_ds = make_cls_dataset(
    train_df,
    training=True,
)

val_cls_ds = make_cls_dataset(
    val_df,
    training=False,
)

test_cls_ds = make_cls_dataset(
    test_df,
    training=False,
)

cls_images, cls_labels = next(
    iter(train_cls_ds)
)

print(
    "Classification image batch:",
    cls_images.shape,
)
print(
    "Pixel range:",
    float(
        tf.reduce_min(
            cls_images
        )
    ),
    "to",
    float(
        tf.reduce_max(
            cls_images
        )
    ),
)
print(
    "Labels:",
    cls_labels.numpy()[:10],
)

## 20. Class weights

In [ ]:
train_labels = (
    train_df["label"]
    .astype(int)
    .values
)

classes = np.unique(
    train_labels
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

CLASS_WEIGHTS = {
    int(class_value): float(weight)
    for class_value, weight
    in zip(classes, weights)
}

print(
    "Training class counts:"
)

display(
    train_df[
        "diagnosis"
    ]
    .value_counts()
    .to_frame("count")
)

print(
    "Class weights:",
    CLASS_WEIGHTS,
)

## 21. Build EfficientNetB0 transfer-learning classifier

In [ ]:
def build_efficientnet_classifier():
    inputs = keras.Input(
        shape=(
            *CLS_IMAGE_SIZE,
            3,
        ),
        name="roi_image",
    )

    x = (
        classification_augmentation(
            inputs
        )
    )

    backbone = (
        keras.applications
        .EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(
                *CLS_IMAGE_SIZE,
                3,
            ),
        )
    )

    backbone._name = (
        "efficientnetb0"
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False,
    )

    x = layers.GlobalAveragePooling2D(
        name="gap"
    )(x)

    x = layers.Dropout(
        0.30,
        name="dropout",
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="prediction",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name=(
            "Melanoma_EfficientNetB0"
        ),
    )


classifier = (
    build_efficientnet_classifier()
)

classifier.compile(
    optimizer=keras.optimizers.Adam(
        CLS_HEAD_LR
    ),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        keras.metrics.AUC(
            name="auc"
        ),
        keras.metrics.Precision(
            name="precision"
        ),
        keras.metrics.Recall(
            name="recall"
        ),
    ],
)

classifier.summary()

dummy_cls = tf.zeros(
    (
        1,
        224,
        224,
        3,
    ),
    dtype=tf.float32,
)

dummy_output = classifier(
    dummy_cls,
    training=False,
)

assert dummy_output.shape == (
    1,
    1,
)

print(
    "EfficientNetB0 classifier "
    "passed the synthetic shape test."
)

## 22. Train classification head

In [ ]:
HEAD_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_head_best.keras"
)

head_callbacks = [
    keras.callbacks.ModelCheckpoint(
        HEAD_MODEL_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

head_history = classifier.fit(
    train_cls_ds,
    validation_data=val_cls_ds,
    epochs=CLS_HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=head_callbacks,
    verbose=1,
)

## 23. Optional fine-tuning

In [ ]:
FINE_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_finetuned_best.keras"
)

fine_history = None

if DO_FINE_TUNING:
    classifier = keras.models.load_model(
        HEAD_MODEL_PATH
    )

    backbone = classifier.get_layer(
        "efficientnetb0"
    )

    backbone.trainable = True

    for layer in backbone.layers[:-20]:
        layer.trainable = False

    for layer in backbone.layers:
        if isinstance(
            layer,
            layers.BatchNormalization,
        ):
            layer.trainable = False

    classifier.compile(
        optimizer=keras.optimizers.Adam(
            CLS_FINE_TUNE_LR
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            keras.metrics.AUC(
                name="auc"
            ),
            keras.metrics.Precision(
                name="precision"
            ),
            keras.metrics.Recall(
                name="recall"
            ),
        ],
    )

    fine_callbacks = [
        keras.callbacks.ModelCheckpoint(
            FINE_MODEL_PATH,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    fine_history = classifier.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_FINE_TUNE_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=fine_callbacks,
        verbose=1,
    )
else:
    print(
        "Fine-tuning disabled."
    )

## 24. Select the classifier using validation ROC-AUC

In [ ]:
def predict_probabilities(
    model,
    dataset,
):
    return (
        model.predict(
            dataset,
            verbose=0,
        )
        .reshape(-1)
    )


head_model = keras.models.load_model(
    HEAD_MODEL_PATH
)

head_val_probs = predict_probabilities(
    head_model,
    val_cls_ds,
)

val_true = (
    val_df["label"]
    .astype(int)
    .values
)

head_val_auc = roc_auc_score(
    val_true,
    head_val_probs,
)

candidate_rows = [
    {
        "Model": (
            "Frozen EfficientNetB0 head"
        ),
        "Path": str(
            HEAD_MODEL_PATH
        ),
        "Validation ROC-AUC": (
            head_val_auc
        ),
    }
]

if (
    DO_FINE_TUNING
    and FINE_MODEL_PATH.exists()
):
    fine_model = (
        keras.models.load_model(
            FINE_MODEL_PATH
        )
    )

    fine_val_probs = (
        predict_probabilities(
            fine_model,
            val_cls_ds,
        )
    )

    fine_val_auc = roc_auc_score(
        val_true,
        fine_val_probs,
    )

    candidate_rows.append(
        {
            "Model": (
                "Fine-tuned EfficientNetB0"
            ),
            "Path": str(
                FINE_MODEL_PATH
            ),
            "Validation ROC-AUC": (
                fine_val_auc
            ),
        }
    )

classifier_comparison_df = (
    pd.DataFrame(
        candidate_rows
    )
)

classifier_comparison_df = (
    classifier_comparison_df
    .sort_values(
        "Validation ROC-AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    classifier_comparison_df
    .round(4)
)

SELECTED_CLASSIFIER_NAME = (
    classifier_comparison_df
    .loc[0, "Model"]
)

SELECTED_CLASSIFIER_PATH = (
    classifier_comparison_df
    .loc[0, "Path"]
)

final_classifier = (
    keras.models.load_model(
        SELECTED_CLASSIFIER_PATH
    )
)

print(
    "Selected classifier:",
    SELECTED_CLASSIFIER_NAME,
)
print(
    "Classification threshold:",
    CLS_THRESHOLD,
)

classifier_comparison_df.to_csv(
    RESULTS_DIR
    / "classifier_validation_comparison.csv",
    index=False,
)

## 25. Classification learning curves

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

head_epochs = range(
    1,
    len(
        head_history.history[
            "loss"
        ]
    ) + 1,
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "loss"
    ],
    label="Head train",
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "val_loss"
    ],
    label="Head val",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "auc"
    ],
    label="Head train",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "val_auc"
    ],
    label="Head val",
)

if fine_history is not None:
    offset = len(
        head_epochs
    )

    fine_epochs = range(
        offset + 1,
        offset
        + len(
            fine_history.history[
                "loss"
            ]
        )
        + 1,
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "loss"
        ],
        label="Fine train",
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "val_loss"
        ],
        label="Fine val",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "auc"
        ],
        label="Fine train",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "val_auc"
        ],
        label="Fine val",
    )

axes[0].set_title(
    "Classification loss"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title(
    "Classification ROC-AUC"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

# PART D — FINAL HELD-OUT TESTING

Only now are the 100 held-out test cases used for final performance reporting.

Final classification results include:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)
- sensitivity
- specificity
- accuracy
- balanced accuracy
- precision
- F1-score
- ROC-AUC

## 26. Final segmentation testing on the 100 held-out cases

In [ ]:
def evaluate_attention_segmentation(
    model,
    frame,
):
    rows = []

    tp_pixels = 0
    tn_pixels = 0
    fp_pixels = 0
    fn_pixels = 0

    for _, row in frame.iterrows():
        image_pil = load_rgb(
            row["image_path"]
        )

        true_mask_pil = load_mask(
            row["mask_path"]
        ).resize(
            (
                SEG_IMAGE_SIZE[1],
                SEG_IMAGE_SIZE[0],
            ),
            Image.Resampling.NEAREST,
        )

        true_mask = (
            np.asarray(true_mask_pil) > 127
        ).astype(np.uint8)

        _, pred_mask = predict_mask_for_pil(
            model,
            image_pil,
        )

        intersection = np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        true_sum = int(
            (true_mask == 1).sum()
        )
        pred_sum = int(
            (pred_mask == 1).sum()
        )

        union = np.logical_or(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        dice = (
            2 * intersection + EPS
        ) / (
            true_sum
            + pred_sum
            + EPS
        )

        iou = (
            intersection + EPS
        ) / (
            union + EPS
        )

        tp = np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()
        tn = np.logical_and(
            true_mask == 0,
            pred_mask == 0,
        ).sum()
        fp = np.logical_and(
            true_mask == 0,
            pred_mask == 1,
        ).sum()
        fn = np.logical_and(
            true_mask == 1,
            pred_mask == 0,
        ).sum()

        tp_pixels += tp
        tn_pixels += tn
        fp_pixels += fp
        fn_pixels += fn

        rows.append(
            {
                "image_id": row["image_id"],
                "diagnosis": row["diagnosis"],
                "image_path": row["image_path"],
                "mask_path": row["mask_path"],
                "dice": float(dice),
                "iou": float(iou),
            }
        )

    per_image_df = pd.DataFrame(rows)

    pixel_sensitivity = (
        tp_pixels
        / (
            tp_pixels
            + fn_pixels
            + EPS
        )
    )

    pixel_specificity = (
        tn_pixels
        / (
            tn_pixels
            + fp_pixels
            + EPS
        )
    )

    return (
        per_image_df,
        float(pixel_sensitivity),
        float(pixel_specificity),
    )


test_eval_dict = selected_segmenter.evaluate(
    test_seg_ds,
    verbose=0,
    return_dict=True,
)

(
    attention_test_per_image_df,
    attention_pixel_sensitivity,
    attention_pixel_specificity,
) = evaluate_attention_segmentation(
    selected_segmenter,
    test_df,
)

test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].mean()
)

test_jaccard = float(
    attention_test_per_image_df[
        "iou"
    ].mean()
)

test_loss = float(
    test_eval_dict["loss"]
)

best_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].max()
)

median_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].median()
)

worst_test_dice = float(
    attention_test_per_image_df[
        "dice"
    ].min()
)

attention_parameters = int(
    selected_segmenter.count_params()
)

attention_segmentation_results = pd.DataFrame(
    {
        "Metric": [
            "Test Dice",
            "Test Jaccard / IoU",
            "Test Loss",
            "Best Test Dice",
            "Median Test Dice",
            "Worst Test Dice",
            "Best Validation Dice",
            "Parameters",
            "Pixel Sensitivity",
            "Pixel Specificity",
        ],
        "Value": [
            test_dice,
            test_jaccard,
            test_loss,
            best_test_dice,
            median_test_dice,
            worst_test_dice,
            best_validation_dice,
            attention_parameters,
            attention_pixel_sensitivity,
            attention_pixel_specificity,
        ],
    }
)

print(
    "ATTENTION U-NET — FINAL HELD-OUT TEST RESULTS"
)
display(
    attention_segmentation_results
)

attention_test_per_image_df.to_csv(
    RESULTS_DIR
    / "attention_unet_test_per_image.csv",
    index=False,
)

attention_segmentation_results.to_csv(
    RESULTS_DIR
    / "attention_unet_final_test_results.csv",
    index=False,
)


## 27. Final classification confusion matrix and clinical metrics

In [ ]:
test_true = (
    test_df["label"]
    .astype(int)
    .values
)

test_probs = (
    final_classifier
    .predict(
        test_cls_ds,
        verbose=0,
    )
    .reshape(-1)
)

test_pred = (
    test_probs
    >= CLS_THRESHOLD
).astype(int)

cm = confusion_matrix(
    test_true,
    test_pred,
    labels=[0, 1],
)

tn, fp, fn, tp = (
    cm.ravel()
)

sensitivity = (
    tp / (tp + fn)
    if (tp + fn)
    else np.nan
)

specificity = (
    tn / (tn + fp)
    if (tn + fp)
    else np.nan
)

accuracy = accuracy_score(
    test_true,
    test_pred,
)

balanced_accuracy = (
    balanced_accuracy_score(
        test_true,
        test_pred,
    )
)

precision = precision_score(
    test_true,
    test_pred,
    zero_division=0,
)

f1 = f1_score(
    test_true,
    test_pred,
    zero_division=0,
)

if len(
    np.unique(test_true)
) == 2:
    roc_auc = roc_auc_score(
        test_true,
        test_probs,
    )
else:
    roc_auc = np.nan

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "True Positive (TP)",
            "True Negative (TN)",
            "False Positive (FP)",
            "False Negative (FN)",
            "Sensitivity",
            "Specificity",
            "Accuracy",
            "Balanced Accuracy",
            "Precision",
            "F1-score",
            "ROC-AUC",
        ],
        "Value": [
            tp,
            tn,
            fp,
            fn,
            sensitivity,
            specificity,
            accuracy,
            balanced_accuracy,
            precision,
            f1,
            roc_auc,
        ],
    }
)

display(
    final_metrics
)

print(
    "\nClassification report:"
)

print(
    classification_report(
        test_true,
        test_pred,
        target_names=[
            "Benign",
            "Malignant",
        ],
        zero_division=0,
    )
)

final_metrics.to_csv(
    RESULTS_DIR
    / "classification_final_test_metrics.csv",
    index=False,
)

test_predictions_df = (
    test_df[
        [
            "image_id",
            "diagnosis",
            "image_path",
            "mask_path",
            "roi_path",
        ]
    ]
    .copy()
)

test_predictions_df[
    "malignancy_probability"
] = test_probs

test_predictions_df[
    "predicted_label"
] = test_pred

test_predictions_df[
    "predicted_diagnosis"
] = np.where(
    test_pred == 1,
    "malignant",
    "benign",
)

test_predictions_df.to_csv(
    RESULTS_DIR
    / "final_test_predictions.csv",
    index=False,
)

# PART E — GRAD-CAM EXPLAINABILITY

Grad-CAM visualises spatial regions that most influenced the classifier output.

It does **not** prove clinical causality and should not be interpreted as a diagnostic explanation.

## 32. Grad-CAM helper

Grad-CAM shows spatial evidence for the *predicted* class on the classifier ROI. The sigmoid logit is used for stable gradients. A positive logit corresponds to melanoma and a negative logit corresponds to benign. A heatmap does not establish clinical causality.

In [ ]:
def make_gradcam_heatmap(model, image_batch, target_class=1):
    backbone = model.get_layer('efficientnetb0')
    gap = model.get_layer('gap')
    dropout = model.get_layer('dropout')
    head = model.get_layer('prediction')
    with tf.GradientTape() as tape:
        feature_maps = backbone(image_batch, training=False)
        tape.watch(feature_maps)
        x = dropout(gap(feature_maps), training=False)
        logits = tf.linalg.matmul(x, head.kernel) + head.bias
        score = logits[:, 0] if target_class == 1 else -logits[:, 0]
    gradients = tape.gradient(score, feature_maps)
    pooled = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.reduce_sum(feature_maps * pooled, axis=-1)[0]
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def prepare_roi_for_classifier(roi_pil):
    resized = roi_pil.convert('RGB').resize(
        (CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    return resized, np.asarray(resized, dtype=np.float32)[None, ...]

def heatmap_overlay(image_pil, heatmap, alpha=0.40):
    image = image_pil.convert('RGB')
    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(
        image.size, Image.Resampling.BILINEAR)
    heat = np.asarray(heat, dtype=np.float32) / 255.0
    base = np.asarray(image, dtype=np.float32) / 255.0
    color = plt.get_cmap('jet')(heat)[..., :3]
    overlay = np.clip((1 - alpha) * base + alpha * color, 0, 1)
    return Image.fromarray(np.uint8(overlay * 255))

# FINAL RESULTS DASHBOARD

This section consolidates the final Attention U-Net segmentation results, training curves, best/median/worst segmentation examples, classifier performance, confusion matrix, ROC-AUC and representative Grad-CAM outputs in one place.


In [ ]:
print("=" * 72)
print("FINAL ATTENTION U-NET SEGMENTATION RESULTS")
print("=" * 72)

final_segmentation_table = pd.DataFrame(
    {
        "Metric": [
            "Test Dice",
            "Test Jaccard / IoU",
            "Test Loss",
            "Best Test Dice",
            "Median Test Dice",
            "Worst Test Dice",
            "Best Validation Dice (hard, batch mean)",
            "Parameters",
        ],
        "Value": [
            test_dice,
            test_jaccard,
            test_loss,
            best_test_dice,
            median_test_dice,
            worst_test_dice,
            best_validation_dice,
            attention_parameters,
        ],
    }
)

display(final_segmentation_table)

# Attention U-Net training curves.
history = attention_history
epochs = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
)

axes[0].plot(epochs, history["loss"], label="Train")
axes[0].plot(
    epochs,
    history["val_loss"],
    linestyle="--",
    label="Validation",
)
axes[0].set_title("Attention U-Net Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(
    epochs,
    history["dice_coefficient"],
    label="Train",
)
axes[1].plot(
    epochs,
    history["val_dice_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[1].set_title("Attention U-Net Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()

axes[2].plot(
    epochs,
    history["iou_coefficient"],
    label="Train",
)
axes[2].plot(
    epochs,
    history["val_iou_coefficient"],
    linestyle="--",
    label="Validation",
)
axes[2].set_title("Attention U-Net IoU / Jaccard")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].legend()

plt.tight_layout()
plt.show()

# Best / Median / Worst segmentation images.
sorted_seg = (
    attention_test_per_image_df
    .sort_values("dice")
    .reset_index(drop=True)
)

worst_row = sorted_seg.iloc[0]
best_row = sorted_seg.iloc[-1]

median_target = float(sorted_seg["dice"].median())
median_idx = (
    sorted_seg["dice"]
    .sub(median_target)
    .abs()
    .idxmin()
)
median_row = sorted_seg.loc[median_idx]

representative_rows = [
    ("Best", best_row),
    ("Median", median_row),
    ("Worst", worst_row),
]

fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 13),
)

for row_idx, (label, result_row) in enumerate(
    representative_rows
):
    image = load_rgb(result_row["image_path"])

    gt_mask_pil = load_mask(
        result_row["mask_path"]
    ).resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0],
        ),
        Image.Resampling.NEAREST,
    )

    gt_mask = (
        np.asarray(gt_mask_pil) > 127
    ).astype(np.uint8)

    _, pred_mask = predict_mask_for_pil(
        selected_segmenter,
        image,
    )

    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(
        f"{label} — Original\n"
        f"{result_row['image_id']}"
    )

    axes[row_idx, 1].imshow(gt_mask, cmap="gray")
    axes[row_idx, 1].set_title("Ground-Truth Mask")

    axes[row_idx, 2].imshow(pred_mask, cmap="gray")
    axes[row_idx, 2].set_title("Predicted Mask")

    axes[row_idx, 3].imshow(
        segmentation_overlay(image, pred_mask)
    )
    axes[row_idx, 3].set_title(
        f"Dice={result_row['dice']:.3f} | "
        f"IoU={result_row['iou']:.3f}"
    )

    for ax in axes[row_idx]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("\n" + "=" * 72)
print("FINAL EFFICIENTNETB0 CLASSIFICATION RESULTS")
print("=" * 72)

classification_dashboard = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Sensitivity",
            "Specificity",
            "Test ROC-AUC",
            "Precision",
            "F1-score",
            "True Positive",
            "True Negative",
            "False Positive",
            "False Negative",
        ],
        "Value": [
            accuracy,
            sensitivity,
            specificity,
            roc_auc,
            precision,
            f1,
            tp,
            tn,
            fp,
            fn,
        ],
    }
)

display(classification_dashboard)

# Confusion matrix.
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(
    ["Predicted Benign", "Predicted Melanoma"]
)
ax.set_yticklabels(
    ["Actual Benign", "Actual Melanoma"]
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=18,
        )

ax.set_title(
    f"Final Test Confusion Matrix (n={len(test_df)})"
)
ax.set_xlabel("Model prediction")
ax.set_ylabel("Ground truth")

plt.tight_layout()
plt.show()

# ROC curve.
if len(np.unique(test_true)) == 2:
    fpr, tpr, _ = roc_curve(
        test_true,
        test_probs,
    )

    plt.figure(figsize=(6, 5))
    plt.plot(
        fpr,
        tpr,
        label=f"EfficientNetB0 AUC = {roc_auc:.3f}",
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title("Final Test ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 72)
print("REPRESENTATIVE GRAD-CAM RESULTS")
print("=" * 72)

# Create the outcome labels needed by the Grad-CAM example selection.
test_predictions_df["outcome"] = [
    ("TP" if true == 1 and pred == 1 else
     "TN" if true == 0 and pred == 0 else
     "FP" if true == 0 and pred == 1 else "FN")
    for true, pred in zip(test_true, test_pred)
]

dashboard_gradcam_rows = []

for outcome in ["TP", "TN", "FP", "FN"]:
    subset = test_predictions_df[
        test_predictions_df["outcome"] == outcome
    ]

    if len(subset):
        dashboard_gradcam_rows.append(
            subset.iloc[0]
        )

for row in dashboard_gradcam_rows:
    roi_pil = load_rgb(
        row["roi_path"]
    )

    roi_resized, roi_batch = (
        prepare_roi_for_classifier(
            roi_pil
        )
    )

    heatmap = make_gradcam_heatmap(
        final_classifier,
        tf.convert_to_tensor(roi_batch),
        target_class=int(row['predicted_label']),
    )

    overlay = heatmap_overlay(
        roi_resized,
        heatmap,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
    )

    axes[0].imshow(
        load_rgb(row["image_path"])
    )
    axes[0].set_title(
        f"Original — {row['outcome']}\n"
        f"True: {row['diagnosis']}"
    )

    axes[1].imshow(roi_resized)
    axes[1].set_title(
        f"Lesion ROI\n"
        f"Pred: {row['predicted_diagnosis']}"
    )

    axes[2].imshow(overlay)
    axes[2].set_title(
        f"Grad-CAM: {row['predicted_diagnosis']} evidence\n"
        f"P(malignant)="
        f"{row['malignancy_probability']:.3f}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

final_segmentation_table.to_csv(
    RESULTS_DIR
    / "FINAL_attention_unet_metrics.csv",
    index=False,
)

classification_dashboard.to_csv(
    RESULTS_DIR
    / "FINAL_classifier_metrics.csv",
    index=False,
)

print(
    "\nFinal result tables saved to:",
    RESULTS_DIR,
)

# PART F — VALIDATION COMPARISON OF CLASSIFIER INPUTS

Run the entire notebook from the top with **Runtime → Run all**. This part is included here; there is no separate `.py` upload.

It compares the original image, the existing predicted-mask ROI, and a new shape-preserving ROI on the *same 400 training and 100 validation cases*. It trains three new classifiers and shows the validation results and threshold tradeoff. The earlier final-test dashboard belongs to the original classifier; it is not a test result for the three new arms.


## 33. Inspect and prepare a new ROI crop

The candidate crops use only predictions from the trained segmenter. The expert masks are shown for visual checking, never used to crop the classifier input.


In [ ]:
from pathlib import Path
from scipy import ndimage
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def candidate_roi(image_pil, mask_binary, padding=0.15):
    image = image_pil.convert("RGB")
    width, height = image.size
    mask = Image.fromarray((np.asarray(mask_binary) > 0).astype("uint8") * 255)
    mask = np.asarray(mask.resize(image.size, Image.Resampling.NEAREST)) > 0
    labelled, n = ndimage.label(mask)
    reason = "cropped"
    if n == 0:
        crop, reason = image, "empty mask"
    else:
        areas = np.bincount(labelled.ravel())
        areas[0] = 0
        component = labelled == areas.argmax()
        area = int(component.sum())
        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:
            crop, reason = image, "implausible mask area"
        else:
            ys, xs = np.where(component)
            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)
            margin = int(np.ceil(span * padding))
            x0 = max(0, int(xs.min()) - margin)
            x1 = min(width, int(xs.max()) + margin + 1)
            y0 = max(0, int(ys.min()) - margin)
            y1 = min(height, int(ys.max()) + margin + 1)
            crop = image.crop((x0, y0, x1, y1))

    # Pad with the crop's edge pixels before resizing, preserving lesion shape.
    a = np.asarray(crop)
    h, w = a.shape[:2]
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    square = np.pad(a, ((top, bottom), (left, right), (0, 0)), mode="edge")
    roi = Image.fromarray(square).resize(
        (CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR
    )
    return roi, reason


candidate_dir = RUNTIME_ROOT / "roi_candidate_validation_experiment"
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_records = []
for split_name, frame in [("train", train_df), ("validation", val_df)]:
    for _, row in frame.iterrows():
        original = load_rgb(row["image_path"])
        _, predicted_mask = predict_mask_for_pil(selected_segmenter, original)
        roi, reason = candidate_roi(original, predicted_mask)
        out = candidate_dir / f"{row['image_id']}_ROI_v2.jpg"
        roi.save(out, quality=95)
        candidate_records.append(
            {"image_id": row["image_id"], "split": split_name,
             "roi_v2_path": str(out), "candidate_reason": reason}
        )

candidate_df = pd.DataFrame(candidate_records).set_index("image_id")
for frame in (train_df, val_df):
    frame["roi_v2_path"] = frame["image_id"].map(candidate_df["roi_v2_path"])
    frame["candidate_reason"] = frame["image_id"].map(candidate_df["candidate_reason"])
    assert frame["roi_v2_path"].notna().all()

print("Existing validation full-image fallbacks:",
      int(val_df["roi_fallback"].sum()), "/", len(val_df))
print("Candidate validation reasons:")
display(val_df["candidate_reason"].value_counts().to_frame("cases"))

# Inspect four of the earlier fallback cases, using the SAME image in each row.
examples = val_df.loc[val_df["roi_fallback"]].head(4)
if len(examples):
    fig, axes = plt.subplots(len(examples), 5, figsize=(16, 3.4 * len(examples)))
    axes = np.atleast_2d(axes)
    for i, (_, row) in enumerate(examples.iterrows()):
        original = load_rgb(row["image_path"])
        expert = np.asarray(load_mask(row["mask_path"])) > 127
        _, predicted = predict_mask_for_pil(selected_segmenter, original)
        panels = [
            original,
            segmentation_overlay(original, expert),
            segmentation_overlay(original, predicted),
            load_rgb(row["roi_path"]),
            load_rgb(row["roi_v2_path"]),
        ]
        titles = [row["image_id"], "Expert overlay", "Predicted overlay",
                  "Old ROI: full image", f"Candidate: {row['candidate_reason']}"]
        for j, (panel, title) in enumerate(zip(panels, titles)):
            axes[i, j].imshow(panel)
            axes[i, j].set_title(title)
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()


## 34. Compare three EfficientNetB0 input choices on validation data

Each choice uses the same split, augmentation, class weights, epoch limits, and validation ROC-AUC selection. Training all three may take some time.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score


def arm_dataset(frame, path_column, training):
    ds = tf.data.Dataset.from_tensor_slices((
        frame[path_column].astype(str).values,
        frame["label"].astype("float32").values,
    ))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_classification_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(CLS_BATCH_SIZE).prefetch(AUTOTUNE)


def fresh_classifier():
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),
        layers.RandomContrast(0.10, seed=SEED),
    ])
    inputs = keras.Input(shape=(*CLS_IMAGE_SIZE, 3))
    backbone = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(*CLS_IMAGE_SIZE, 3)
    )
    backbone.trainable = False
    x = backbone(augmentation(inputs), training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs), backbone


def compile_classifier(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )


def fit_one_arm(name, path_column):
    keras.utils.set_random_seed(SEED)
    train_ds = arm_dataset(train_df, path_column, training=True)
    val_ds = arm_dataset(val_df, path_column, training=False)
    y = val_df["label"].astype(int).to_numpy()
    model, backbone = fresh_classifier()
    compile_classifier(model, CLS_HEAD_LR)
    model.fit(
        train_ds, validation_data=val_ds, epochs=CLS_HEAD_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=5,
            restore_best_weights=True)],
        verbose=1,
    )
    head_prob = model.predict(val_ds, verbose=0).ravel()
    head_auc = roc_auc_score(y, head_prob)
    head_weights = model.get_weights()
    chosen_stage, prob, auc = "frozen head", head_prob, head_auc

    if DO_FINE_TUNING:
        backbone.trainable = True
        for layer in backbone.layers[:-20]:
            layer.trainable = False
        for layer in backbone.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False
        compile_classifier(model, CLS_FINE_TUNE_LR)
        model.fit(
            train_ds, validation_data=val_ds, epochs=CLS_FINE_TUNE_EPOCHS,
            class_weight=CLASS_WEIGHTS,
            callbacks=[keras.callbacks.EarlyStopping(
                monitor="val_auc", mode="max", patience=4,
                restore_best_weights=True)],
            verbose=1,
        )
        fine_prob = model.predict(val_ds, verbose=0).ravel()
        fine_auc = roc_auc_score(y, fine_prob)
        if fine_auc > head_auc:
            chosen_stage, prob, auc = "fine-tuned", fine_prob, fine_auc
        else:
            model.set_weights(head_weights)

    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    metrics = {
        "Input": name, "Stage": chosen_stage, "Validation AUC": auc,
        "Accuracy at 0.5": (tp + tn) / len(y),
        "Sensitivity at 0.5": tp / (tp + fn),
        "Specificity at 0.5": tn / (tn + fp),
        "FP": int(fp), "FN": int(fn),
    }
    return metrics, model, prob


arms = [
    ("Full image", "image_path"),
    ("Existing ROI", "roi_path"),
    ("Candidate ROI", "roi_v2_path"),
]
experiment_models, experiment_probs, comparison_rows = {}, {}, []
for name, column in arms:
    print("\nTraining:", name)
    metrics, model, probabilities = fit_one_arm(name, column)
    comparison_rows.append(metrics)
    experiment_models[name] = model
    experiment_probs[name] = probabilities

comparison = pd.DataFrame(comparison_rows).sort_values(
    "Validation AUC", ascending=False
)
display(comparison.round(3))
comparison.to_csv(RESULTS_DIR / "roi_input_validation_comparison.csv", index=False)


## 35. See the sensitivity and specificity tradeoff

These are **validation** results. Choosing a lower threshold often catches more melanomas and increases false positives. Choose a configuration on validation data before a separate final assessment.


In [ ]:
selected_input = comparison.iloc[0]["Input"]
selected_prob = experiment_probs[selected_input]
y_val = val_df["label"].astype(int).to_numpy()
threshold_rows = []
for cutoff in np.arange(0.10, 0.91, 0.05):
    pred = (selected_prob >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    threshold_rows.append({
        "Threshold": round(float(cutoff), 2),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Accuracy": (tp + tn) / len(y_val),
        "FP": int(fp), "FN": int(fn),
    })

print("Validation threshold tradeoff for:", selected_input)
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))
threshold_table.to_csv(RESULTS_DIR / "roi_input_validation_thresholds.csv", index=False)
print("Choose a model and threshold using validation only; do not tune on test_df.")


# Limitations and next steps

The 100-case test set was used for the *original* ROI classifier. The three new input choices in Part F were assessed on validation data only. Their comparison must not be reported as new test accuracy or evidence of a proven improvement. Choose a model and decision threshold using validation data, then assess once on a separate independent test group.

Image-ID separation does not exclude repeat images from the same patient. The CSV has no patient identifiers. Performance on this dermoscopic dataset may not generalise to smartphone images or other skin-tone groups. Grad-CAM is a visualisation of model influence, not a clinical explanation.


# 37. References

1. Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC).* arXiv:1605.01397.

2. Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.

3. Oktay, O., et al. (2018). *Attention U-Net: Learning Where to Look for the Pancreas.* arXiv:1804.03999.

4. Tan, M., & Le, Q. V. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML.

5. Selvaraju, R. R., et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.* ICCV.

6. International Skin Imaging Collaboration (ISIC), **2016 Task 3B: Segmented Lesion Classification**.

# Workflow

900 ISIC 2016 labels → 400 train / 100 validation / 100 original final-test images → Attention U-Net → predicted masks and ROIs → EfficientNetB0 → original pipeline test metrics and Grad-CAM → three new input choices compared on validation data.

**Report the Part F values as validation results only.** A new independent test is needed to assess any chosen change.
